In [ ]:
import cv2
import numpy as np
import os

input_folder = "input"
output_folder = "output"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):

    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_folder, filename)
    image = cv2.imread(image_path)

    if image is None:
        continue

    image = cv2.resize(image, (800, 800))

    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

   
    median_val = np.median(blur)
    lower = int(max(0, 0.66 * median_val))
    upper = int(min(255, 1.33 * median_val))

    edges = cv2.Canny(blur, lower, upper)

    
    kernel = np.ones((5, 5), np.uint8)

   
    edges = cv2.dilate(edges, kernel, iterations=2)
    edges = cv2.erode(edges, kernel, iterations=1)

    
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)


    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    intact = 0
    broken = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area < 500 or area > 200000:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        if w > 750 or h > 750:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue


        circularity = 4 * np.pi * area / (perimeter * perimeter)

        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)

        if hull_area == 0:
            continue

        solidity = float(area) / hull_area


        if circularity > 0.75 and solidity > 0.50:
            label = "Intact Biscuit"
            color = (0, 255, 0)
            intact += 1
        else:
            label = "Broken Biscuit"
            color = (0, 0, 255)
            broken += 1

       
        cv2.drawContours(image, [cnt], -1, color, 2)
        cv2.rectangle(image, (x, y), (x + w, y + h), color, 2)

        cv2.putText(image, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)


  
    output_path = os.path.join(output_folder, f"processed_{filename}")
    cv2.imwrite(output_path, image)

print("Outline-based processing complete. Check output_images folder.")

Outline-based processing complete. Check output_images folder.
